# WAI-illustrious SDXL — tạo ảnh trên Google Colab

Notebook chạy **checkpoint SDXL đầy đủ** (`.safetensors`) bằng Diffusers, không cần WebUI/link chia sẻ công khai. Nếu chưa có model, ô 4 **tự tải WAI-illustrious v17** từ [bản lưu trên Hugging Face](https://huggingface.co/LyliaEngine/waiIllustriousSDXL_v170/blob/32be7bfdcd406db70df663b9cee3313957deb68f/waiIllustriousSDXL_v170.safetensors). File ~6,94 GB; SHA-256 được ghim và đối chiếu với [metadata bản gốc trên Civitai](https://civitai.com/api/v1/model-versions/2883731). Không cần token, không tải thêm trọng số SDXL base.

**Bắt đầu:**
1. Vào **Runtime → Change runtime type → GPU** (T4 hoặc GPU mạnh hơn).
2. Chạy lần lượt các ô **1 → 6**, cho phép gắn Google Drive. Chưa có file? `AUTO_DOWNLOAD=True` (mặc định) sẽ tự tải và, nếu có thể, lưu lại vào `MyDrive/AI/models/WAI-illustrious.safetensors` cho lần sau. **Đã có checkpoint riêng?** Sửa `MODEL_PATH` ở ô 3; file có sẵn luôn được ưu tiên, không bị thay thế.
3. Sửa prompt ở ô 6 rồi chạy lại ô 6 để tạo ảnh mới. PNG mặc định lưu vào `MyDrive/AI/outputs`.

Cần khoảng 7–9 GiB đĩa trống để tải model (nếu ổ `/content` ít chỗ, notebook thử tải trực tiếp vào Drive). Cấu hình/tokenizer SDXL được Diffusers tự tải khi nạp model lần đầu. GPU/RAM và hạn mức sử dụng phụ thuộc Colab; notebook không thể tự cấp thêm RAM hay GPU cho tài khoản.

In [ ]:
# @title 1. Kiểm tra GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Chọn Runtime → Change runtime type → GPU, rồi chạy lại ô này.")
device = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"GPU: {device.name} | VRAM trống: {free_bytes / 2**30:.1f}/{total_bytes / 2**30:.1f} GiB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# @title 2. Tự cài thư viện còn thiếu (giữ PyTorch/CUDA của Colab)
%pip -q install "diffusers==0.35.2" "transformers==4.52.4" "accelerate==1.10.1" "safetensors>=0.4.5,<1" "huggingface-hub==0.36.2" "hf-xet>=1.1.3,<2"

In [ ]:
# @title 3. Cấu hình model, Drive và bộ nhớ { display-mode: "form" }
MOUNT_DRIVE = True # @param {type:"boolean"}
MODEL_PATH = "/content/drive/MyDrive/AI/models/WAI-illustrious.safetensors" # @param {type:"string"}
AUTO_DOWNLOAD = True # @param {type:"boolean"}
PERSIST_MODEL_TO_DRIVE = True # @param {type:"boolean"}
CACHE_MODEL_LOCAL = True # @param {type:"boolean"}
OUTPUT_DIR = "/content/drive/MyDrive/AI/outputs" # @param {type:"string"}
VRAM_MODE = "auto" # @param ["auto", "speed", "low_vram"]

from pathlib import Path

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

source_model = Path(MODEL_PATH).expanduser()
output_dir = Path(OUTPUT_DIR).expanduser()
local_cache_root = Path("/content/wai_model_cache")
drive_root = Path("/content/drive")
if not source_model.is_absolute() or not output_dir.is_absolute():
    raise ValueError("MODEL_PATH và OUTPUT_DIR phải là đường dẫn tuyệt đối.")
if source_model.suffix.lower() != ".safetensors":
    raise ValueError("MODEL_PATH phải kết thúc bằng .safetensors (checkpoint đầy đủ, không phải LoRA).")
using_drive = any(drive_root == p or drive_root in p.parents for p in (source_model, output_dir))
if using_drive and not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive chưa được gắn. Bật MOUNT_DRIVE rồi chạy lại ô 3, hoặc đổi đường dẫn sang /content.")
if VRAM_MODE not in ("auto", "speed", "low_vram"):
    raise ValueError("VRAM_MODE phải là auto, speed hoặc low_vram.")
print("Model có sẵn:" if source_model.is_file() else "Model chưa có (sẽ tự tải ở ô 4):", source_model)
print("Thư mục lưu ảnh:", output_dir)

In [ ]:
# @title 4. Tự tải/checkpoint: ưu tiên file có sẵn, kiểm tra SHA-256 khi tải mới
import hashlib
import os
import shutil
# Tăng thời gian chờ trên mạng Colab chập chờn; đặt trước khi import huggingface_hub.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")
from huggingface_hub import hf_hub_download
from safetensors import safe_open

# Bản v17 pruned FP16; hash trùng file Civitai model version 2883731.
HF_REPO = "LyliaEngine/waiIllustriousSDXL_v170"
HF_FILENAME = "waiIllustriousSDXL_v170.safetensors"
HF_REVISION = "32be7bfdcd406db70df663b9cee3313957deb68f"
HF_SHA256 = "f116b0c78ff441467b0cdc8f1936e1ed18ea31e9997c7b132b1b8db533f0bd04"
HF_MODEL_BYTES = 6_938_040_682
MIN_CHECKPOINT_BYTES = 100 * 2**20
DISK_RESERVE_BYTES = 2 * 2**30

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 2**20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def inspect_checkpoint(path, verify_official=False):
    path = Path(path)
    if path.suffix.lower() != ".safetensors" or not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy checkpoint .safetensors: {path}")
    size = path.stat().st_size
    if size < MIN_CHECKPOINT_BYTES:
        raise ValueError("File model quá nhỏ: có thể tải dở, là HTML hoặc là LoRA.")
    if verify_official:
        if size != HF_MODEL_BYTES or sha256_file(path) != HF_SHA256:
            raise ValueError("SHA-256/dung lượng model tải về không trùng bản v17 gốc; không nạp file này.")
    try:
        with safe_open(str(path), framework="pt", device="cpu") as header:
            keys = header.keys()  # chỉ đọc header, không tải trọng số vào RAM
            has_unet = any(key.startswith("model.diffusion_model.") for key in keys)
            has_clip = any(key.startswith("conditioner.embedders.") for key in keys)
    except Exception as exc:
        raise ValueError("Không đọc được safetensors; hãy kiểm tra file checkpoint.") from exc
    if not (has_unet and has_clip):
        raise ValueError("Cần checkpoint SDXL đầy đủ (UNet + text encoder), không phải LoRA/UNet-only.")
    return size

def copy_atomic(source, destination, size, preserve_mtime=False):
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".partial")
    try:
        # Drive FUSE có thể không cho phép copy2/copystat; chỉ giữ mtime khi cache vào /content.
        if preserve_mtime:
            shutil.copy2(source, partial)
        else:
            shutil.copyfile(source, partial)
        if partial.stat().st_size != size:
            raise OSError("Bản sao checkpoint không đầy đủ.")
        os.replace(partial, destination)
    finally:
        partial.unlink(missing_ok=True)

if source_model.exists() and not source_model.is_file():
    raise ValueError(f"MODEL_PATH là thư mục hoặc tệp đặc biệt, không phải checkpoint: {source_model}")
if source_model.is_file():
    model_size = inspect_checkpoint(source_model)  # file của người dùng có thể là phiên bản khác
    checkpoint = source_model
    if CACHE_MODEL_LOCAL and drive_root in source_model.parents:
        local_cache_root.mkdir(parents=True, exist_ok=True)
        cached = local_cache_root / source_model.name
        same_file = (
            cached.is_file()
            and cached.stat().st_size == model_size
            and cached.stat().st_mtime_ns == source_model.stat().st_mtime_ns
        )
        if same_file:
            checkpoint = cached
            print("Dùng lại bản sao /content trong phiên này.")
        elif shutil.disk_usage(local_cache_root).free >= model_size + DISK_RESERVE_BYTES:
            try:
                print("Sao chép model từ Drive sang /content để nạp nhanh hơn...")
                copy_atomic(source_model, cached, model_size, preserve_mtime=True)
                checkpoint = cached
            except OSError as exc:
                print(f"Không sao chép được ({type(exc).__name__}); nạp trực tiếp từ Drive.")
        else:
            print("Đĩa /content không đủ chỗ để sao chép; nạp trực tiếp từ Drive.")
else:
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f"Chưa có model: {source_model}. Bật AUTO_DOWNLOAD ở ô 3 hoặc đặt checkpoint vào đúng đường dẫn.")
    local_cache_root.mkdir(parents=True, exist_ok=True)
    disk_ok = shutil.disk_usage(local_cache_root).free >= HF_MODEL_BYTES + DISK_RESERVE_BYTES
    on_drive = drive_root in source_model.parents
    print(f"Chưa có checkpoint; đang tải bản WAI-illustrious v17 ({HF_MODEL_BYTES / 10**9:.2f} GB)...")
    if disk_ok:
        try:
            downloaded = Path(hf_hub_download(
                repo_id=HF_REPO, filename=HF_FILENAME, revision=HF_REVISION,
                cache_dir=str(local_cache_root), token=False,
            ))
        except Exception as exc:
            raise RuntimeError("Không tải được model từ Hugging Face. Kiểm tra Internet hoặc tự đặt file ở MODEL_PATH.") from exc
        inspect_checkpoint(downloaded, verify_official=True)
        checkpoint = downloaded  # không sao chép lại sang /content lần thứ hai
        if on_drive and PERSIST_MODEL_TO_DRIVE:
            try:
                print("Lưu bản đã xác minh vào Drive để phiên sau không tải lại...")
                copy_atomic(downloaded, source_model, HF_MODEL_BYTES)
            except OSError as exc:
                print(f"Drive không lưu được ({type(exc).__name__}); vẫn tạo ảnh từ bản tạm ở /content.")
    elif on_drive and PERSIST_MODEL_TO_DRIVE:
        # Đĩa máy Colab quá ít: tải vào Drive, không tạo bản sao 7 GB trên /content.
        source_model.parent.mkdir(parents=True, exist_ok=True)
        try:
            downloaded = Path(hf_hub_download(
                repo_id=HF_REPO, filename=HF_FILENAME, revision=HF_REVISION,
                local_dir=str(source_model.parent), token=False,
            ))
            inspect_checkpoint(downloaded, verify_official=True)
            if downloaded != source_model:
                os.replace(downloaded, source_model)
            checkpoint = source_model
        except Exception as exc:
            raise RuntimeError("Không đủ đĩa /content và tải vào Drive không thành công. Kiểm tra dung lượng/quyền Drive.") from exc
    else:
        raise OSError("Thiếu đĩa trống để tải model (~9 GiB). Giải phóng đĩa, hoặc bật lưu vào Drive ở ô 3.")
    print("Đã đối chiếu SHA-256 với bản v17 gốc.")
print(f"Sẵn sàng: {checkpoint} ({checkpoint.stat().st_size / 2**30:.2f} GiB)")

In [ ]:
# @title 5. Nạp model, tự chọn GPU nhanh / CPU offload khi thiếu VRAM
import gc
import psutil
from diffusers import EulerAncestralDiscreteScheduler, StableDiffusionXLPipeline

if "pipe" in globals():
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
ram_gib = psutil.virtual_memory().available / 2**30
print(f"RAM hệ thống khả dụng: {ram_gib:.1f} GiB")
if ram_gib < 8:
    print("Cảnh báo: nạp checkpoint 6,94 GB có thể cần Colab high-RAM nếu phiên này quá ít RAM.")

torch.backends.cuda.matmul.allow_tf32 = True  # tăng tốc nếu GPU hỗ trợ TF32
torch.backends.cudnn.allow_tf32 = True
free_bytes, _ = torch.cuda.mem_get_info()
use_offload = VRAM_MODE == "low_vram" or (VRAM_MODE == "auto" and free_bytes < 14 * 2**30)

def create_pipeline(offload):
    pipeline = StableDiffusionXLPipeline.from_single_file(
        str(checkpoint), torch_dtype=torch.float16, use_safetensors=True,
    )
    pipeline.scheduler = EulerAncestralDiscreteScheduler.from_config(pipeline.scheduler.config)
    pipeline.vae.enable_slicing()
    if offload:
        pipeline.vae.enable_tiling()
        pipeline.enable_model_cpu_offload()
    else:
        pipeline.to("cuda")
    return pipeline

print("Chế độ:", "tiết kiệm VRAM (CPU offload)" if use_offload else "ưu tiên tốc độ (GPU FP16)")
gpu_oom = False
try:
    pipe = create_pipeline(use_offload)
except torch.cuda.OutOfMemoryError:
    gpu_oom = True
if gpu_oom:
    if VRAM_MODE != "auto" or use_offload:
        raise RuntimeError("Hết VRAM khi nạp model. Chọn VRAM_MODE='low_vram', rồi chạy lại ô 5.")
    gc.collect()
    torch.cuda.empty_cache()
    print("Không đủ VRAM để nạp trực tiếp; đang tự thử CPU offload...")
    pipe = create_pipeline(True)
    use_offload = True
print("Model đã sẵn sàng. Chạy ô 6 để tạo ảnh.")

In [ ]:
# @title 6. Tạo ảnh (sửa prompt/seed rồi chạy lại tùy thích) { display-mode: "form" }
PROMPT = "general, 1girl, solo, cherry blossoms, spring, soft sunlight, detailed eyes, anime illustration, masterpiece, best quality" # @param {type:"string"}
NEGATIVE_PROMPT = "nsfw, explicit, lowres, worst quality, bad anatomy, blurry" # @param {type:"string"}
SIZE = "1024x1024" # @param ["1024x1024", "832x1216", "1216x832", "768x1024", "1024x768", "1024x1344", "1344x1024"]
STEPS = 25 # @param {type:"slider", min:10, max:45, step:1}
CFG = 6.0 # @param {type:"slider", min:1, max:12, step:0.5}
SEED = -1 # @param {type:"integer"}
EMBED_METADATA = True # @param {type:"boolean"}

import json
import secrets
from datetime import datetime, timezone
from IPython.display import display
from PIL.PngImagePlugin import PngInfo

if "pipe" not in globals():
    raise RuntimeError("Chưa nạp model. Hãy chạy ô 5 trước.")
if not PROMPT.strip():
    raise ValueError("PROMPT không được để trống.")
width, height = map(int, SIZE.split("x"))
if min(width, height) < 512 or width % 8 or height % 8:
    raise ValueError("Kích thước ảnh phải >= 512 và chia hết cho 8.")
if not 1 <= STEPS <= 50 or not 1 <= CFG <= 12:
    raise ValueError("STEPS phải từ 1–50 và CFG từ 1–12.")
if SEED != -1 and not 0 <= SEED < 2**32:
    raise ValueError("SEED phải là -1 (ngẫu nhiên) hoặc số nguyên từ 0 đến 2^32 - 1.")
seed = secrets.randbelow(2**32) if SEED == -1 else SEED

def run_generation():
    generator = torch.Generator(device="cpu").manual_seed(seed)
    with torch.inference_mode():
        return pipe(
            prompt=PROMPT.strip(), negative_prompt=NEGATIVE_PROMPT.strip(),
            width=width, height=height, num_inference_steps=STEPS,
            guidance_scale=CFG, num_images_per_prompt=1, generator=generator,
        ).images[0]

gpu_oom = False
try:
    image = run_generation()
except torch.cuda.OutOfMemoryError:
    gpu_oom = True
if gpu_oom:
    torch.cuda.empty_cache()
    if VRAM_MODE == "auto" and not use_offload:
        print("Hết VRAM khi tạo ảnh; đang tự chuyển sang CPU offload và thử lại cùng seed...")
        del pipe
        gc.collect()
        torch.cuda.empty_cache()
        pipe = create_pipeline(True)
        use_offload = True
        try:
            image = run_generation()
        except torch.cuda.OutOfMemoryError as exc:
            torch.cuda.empty_cache()
            raise RuntimeError("Vẫn thiếu VRAM. Chọn kích thước 768x1024 hoặc 1024x768 rồi chạy lại ô 6.") from exc
    else:
        raise RuntimeError("Hết VRAM. Giảm kích thước, hoặc đặt VRAM_MODE='low_vram' và chạy lại ô 5–6.")

display(image)
png_info = PngInfo()
if EMBED_METADATA:
    png_info.add_text("parameters", json.dumps({
        "model": checkpoint.name,
        "prompt": PROMPT.strip(), "negative_prompt": NEGATIVE_PROMPT.strip(),
        "seed": seed, "width": width, "height": height,
        "steps": STEPS, "cfg": CFG, "sampler": "Euler a",
    }, ensure_ascii=False))
filename = f"wai_{datetime.now(timezone.utc):%Y%m%d_%H%M%S_%f}_{seed}.png"

def save_image(directory):
    directory.mkdir(parents=True, exist_ok=True)
    destination = directory / filename
    partial = directory / (filename + ".partial")
    try:
        image.save(partial, format="PNG", pnginfo=png_info)
        os.replace(partial, destination)
    finally:
        partial.unlink(missing_ok=True)
    return destination

try:
    if (output_dir == drive_root or drive_root in output_dir.parents) and not Path("/content/drive/MyDrive").is_dir():
        raise OSError("Drive đã ngắt kết nối")
    output_path = save_image(output_dir)
except OSError as exc:
    backup_dir = Path("/content/wai_outputs")
    if output_dir == backup_dir:
        raise
    print(f"Không lưu được vào {output_dir} ({type(exc).__name__}); lưu tạm ở {backup_dir}.")
    output_path = save_image(backup_dir)
print(f"Seed: {seed} | Đã lưu: {output_path}")

### Mẹo và xử lý sự cố

- **Tải tự động:** nếu `MODEL_PATH` chưa tồn tại và `AUTO_DOWNLOAD=True`, notebook tải **bản v17 pruned FP16** (không phải FP8/LoRA), kiểm tra kích thước + SHA-256 trước khi nạp. Mạng phải truy cập được Hugging Face. File có sẵn tại `MODEL_PATH` **không bị ghi đè**; checkpoint của bạn có thể là phiên bản khác v17.
- **Lưu model lâu dài:** nếu `MODEL_PATH` ở Drive và `PERSIST_MODEL_TO_DRIVE=True`, bản tải về sẽ được chép vào Drive. Nếu Drive hết quota, notebook vẫn chạy từ bản tạm trong `/content` khi tải tại đó thành công. Nếu ổ `/content` không đủ chỗ, notebook thử tải thẳng vào Drive. `/content` bị xóa khi phiên Colab kết thúc.
- **Thiếu đĩa/RAM/VRAM:** `CACHE_MODEL_LOCAL=True` chỉ sao chép file Drive có sẵn khi còn đủ đĩa, nếu không sẽ nạp trực tiếp từ Drive. `VRAM_MODE=auto` dùng GPU khi VRAM trống ≥ 14 GiB và tự thử offload khi OOM; `low_vram` ép CPU offload + VAE tiling (chậm hơn). Nếu vẫn OOM, dùng ảnh nhỏ hơn; `1024x1344`/`1344x1024` phù hợp khi GPU có nhiều VRAM và cần khung hình lớn hơn. Nếu thiếu **RAM hệ thống** khi nạp, cần runtime high-RAM/nhẹ hơn. Notebook không thể tự cấp GPU/RAM.
- **Sai file:** LoRA, file tải chưa xong, hoặc FP8 chỉ dành cho công cụ khác sẽ không dùng được. Cấu hình/tokenizer SDXL do Diffusers tải thêm khi cần; không tải thêm trọng số SDXL base. Nếu Google Drive ngắt khi lưu ảnh, notebook lưu dự phòng ở `/content/wai_outputs` — hãy tải ảnh xuống trước khi ngắt phiên.
- `SEED=-1` chọn seed ngẫu nhiên; nhập seed cụ thể để thử lặp lại cùng prompt/cấu hình. Sinh **một ảnh mỗi lượt** để giảm đỉnh VRAM. PNG có thể chứa prompt trong metadata nếu `EMBED_METADATA=True`; cân nhắc tắt trước khi chia sẻ. Prompt mặc định hướng tới nội dung lành mạnh nhưng **không đảm bảo lọc nội dung**; hãy kiểm tra ảnh và tuân thủ điều khoản sử dụng model/Colab.